# 06 · From classification to action and reporting — Neonatal mortality NCHS (M4)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 → 02: S4 · G1 → 03: S5 · S6 · S7 · G2 · G3 → 04: S8 → 05: S8 · S9 · G4 → **06: S10 · S11** |
| What it does | Turns the **predictor** class into a use: workload at several operating points, equity by subgroup, the original objective of the thesis and what drives the risk |
| Stages | E10 · E11 |
| Design | Model fitted once on 2016–2021 and applied to the temporal test 2022–2023, as the births it would meet in practice |
| Inputs | Contract table (notebook 03) and `data/outputs_04_selection.json` |
| Outputs | `data/outputs_06_e10_action.txt`, `figures/06_*.png`; guide: `docs/e10_action_neonatal.md` |
| Time | 20 to 40 minutes |
| Previous | `05_main_evaluation` |

In [ ]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.optimize import brentq
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)      # decision texts are shown in full
import warnings
# scikit-learn >= 1.8 announces that LogisticRegression(penalty=...) will be replaced by l1_ratio. The models keep
# penalty (the hyperparameters chosen in notebook 04) so that results stay exactly as reported; the notice is hidden.
warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*", category=FutureWarning)
lines_es = []
if LANG == "es":                       # progress messages (print) also in Spanish
    import builtins
    def print(*args, **kwargs):
        builtins.print(*[translate(a, "es") if isinstance(a, str) and not a.startswith("\n== ") else a for a in args], **kwargs)

def write_outputs(path, header):
    # English text output (cited by the documents) and, from the Spanish notebook, a Spanish copy (*_es.txt).
    path.write_text(header + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
    if LANG == "es":
        path.with_name(path.stem + "_es.txt").write_text(translate(header, "es") + "\n" + "\n".join(lines_es) + "\n", encoding="utf-8")

CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "neonatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_06_e10_action.txt"
SELECTION_FILE = ROOT / "data" / "outputs_04_selection.json"
SEED = 20261123
DEV_YEARS, TEST_YEARS = list(range(2016, 2022)), [2022, 2023]
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))
QUICK = os.environ.get("NCHS_QUICK") == "1"
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
B = 30 if QUICK else 200
META = ["unit_id", "group_id", "period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal",
        "ge28", "ga_imputed", "bw_imputed", "n_unknown"]
PREDICTORS = [c for c in pq.read_schema(CONTRACT_DIR / "cohort_2021.parquet").names if c not in META]
REFERENCE = ["gestational_age", "birthweight"]           # reference model fixed in the outcome card
EXTRA = ["period", "label_main", "at_risk_48h", "label_day2_27", "label_early_neonatal", "ge28", "n_unknown"]

# Primary learner and hyperparameters chosen by the preliminary run (notebook 04, selection rule)
sel = json.loads(SELECTION_FILE.read_text(encoding="utf-8"))
assert QUICK or not sel["quick"], "notebook 04 was run in QUICK mode: run it again with the full sample"
primary, BEST = sel["primary"], sel["best"]

def make_model(name):
    cfg = dict(BEST[name]); n_iter = cfg.pop("n_estimators", None)
    imp = SimpleImputer(strategy="median")                       # decision M4-D3: no missingness indicators
    if name == "LR":
        return make_pipeline(imp, StandardScaler(), LogisticRegression(solver="liblinear", max_iter=500, **cfg))
    n_trees = 50 if QUICK else 300
    if name == "RF":
        return make_pipeline(imp, RandomForestClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    if name == "ET":
        return make_pipeline(imp, ExtraTreesClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    return make_pipeline(imp, XGBClassifier(n_estimators=(min(n_iter, 50) if QUICK else n_iter), tree_method="hist",
                                            n_jobs=N_JOBS, random_state=SEED, **cfg))

pd.set_option("display.max_colwidth", None)
lines = []
def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)
SUPPRESS_SMALL = False
import contextlib, io, html as _html
_show_en = show

def _fmt(v):
    # Display format of a table cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if SUPPRESS_SMALL and isinstance(v, (int, np.integer)) and 0 < v < 10:
        return "<10"
    if isinstance(v, (int, np.integer)) or (isinstance(v, (float, np.floating)) and float(v).is_integer() and abs(v) >= 1000):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _render(title, obj):
    # In Jupyter: a heading and an HTML table with scroll; elsewhere: the text block.
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display as _display
        if get_ipython() is None:              # plain Python (not a notebook)
            raise ImportError
    except ImportError:
        return print(lines_es[-1] if LANG == "es" else lines[-1])
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(str(title))}</b></p>"
    if isinstance(obj, (pd.Series, pd.DataFrame)):
        if isinstance(obj, pd.Series):          # rows are different quantities: each cell on its own
            df = obj.astype(object).map(_fmt).to_frame(name=translate("value", LANG))
        else:                                    # columns: the same number of decimals down each column
            df = obj.copy().astype(object)
            for k in range(df.shape[1]):
                col = obj.iloc[:, k]
                if pd.api.types.is_float_dtype(col) and not col.dropna().map(float.is_integer).all():
                    dec = max((len(f"{v:.4g}".split(".")[1]) if "." in f"{v:.4g}" and "e" not in f"{v:.4g}" else 0)
                              for v in col.dropna()) if col.notna().any() else 0
                    df.iloc[:, k] = col.map(lambda v, d=dec: "" if pd.isna(v) else f"{v:,.{d}f}")
                else:
                    df.iloc[:, k] = col.map(_fmt)
        table = df.to_html(na_rep="", border=0)
        _display(HTML(head + f"<div style='max-height:420px; overflow:auto'>{table}</div>"))
    else:
        _display(HTML(head + f"<pre style='margin:0'>{_html.escape(str(obj))}</pre>"))

def show(title, obj):
    # The English block is kept in `lines` (data/outputs_*.txt) and, in Spanish, the translated block in
    # `lines_es` (data/outputs_*_es.txt); on screen the table is shown as an HTML table.
    with contextlib.redirect_stdout(io.StringIO()):
        _show_en(title, obj)
        if LANG == "es":
            _show_en(translate(title, "es"), translate(obj, "es"))
            lines_es.append(lines.pop())
    _render(translate(title, LANG), translate(obj, LANG))
show("Primary learner (notebook 04)", pd.Series({"primary learner": primary, "hyperparameters": json.dumps(BEST[primary]),
                                                  "ensemble adopted in notebook 04": sel["ensemble_adopted"]}))

## E10 · From classification to action — samples

Same samples as notebook 05.

In [ ]:
# Samples: every event and FRACTION of non-events, weight 1/FRACTION for non-events (the weighted
# sample stands for the full population). Development 2016-2021; temporal test 2022-2023.
def load(years, seed):
    rng = np.random.default_rng(seed); parts = []; n = e = 0
    for y in years:     # read in batches and keep the sample only, so the memory stays low
        for b in pq.ParquetFile(CONTRACT_DIR / f"cohort_{y}.parquet").iter_batches(500_000, columns=PREDICTORS + EXTRA):
            c = b.to_pandas()
            n += len(c); e += int(c["label_main"].sum())
            parts.append(c[c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)])
    d = pd.concat(parts, ignore_index=True)
    if QUICK:
        d = d.sample(n=min(len(d), 40000), random_state=seed).reset_index(drop=True)
    d["w"] = np.where(d["label_main"], 1.0, 1.0 / FRACTION)
    return d, n, e

dev, n_dev, e_dev = load(DEV_YEARS, SEED)
test, n_test, e_test = load(TEST_YEARS, SEED + 1)
show("Data", pd.DataFrame({"births": [n_dev, n_test], "neonatal deaths": [e_dev, e_test],
                           "sample rows": [len(dev), len(test)]}, index=["development 2016-2021", "temporal test 2022-2023"]))

## E10 · From classification to action — model applied to the temporal test

Check: the AUROC must match the temporal validation of notebook 05.

| Result | What is done | In this model |
|---|---|---|
| AUROC = that of notebook 05 | Continue | 0,961 |
| Different | Check that the same selection and sample are used | — |

In [ ]:
# Model of stage S10: the primary learner fitted once on the whole development sample (2016-2021) and
# recalibrated inside it, applied once to the temporal test (2022-2023), as in notebook 05 (criterion C4).
# The test cohorts play the role of the births the model would meet in practice.
# Metrics (weighted where the sampling matters; AUROC does not depend on it)
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
expit = lambda z: 1 / (1 + np.exp(-z))

def cal_slope_intercept(y, p, w):
    lp = logit(p)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y, sample_weight=w).coef_[0, 0]
    f = lambda a: np.sum(w * (y - expit(a + lp)))        # calibration-in-the-large (offset = lp)
    return float(slope), float(brentq(f, -10, 10))

def metrics(y, p, w):
    o = np.argsort(p); cw = np.cumsum(w[o]); m = {}
    for share in (0.05, 0.10):
        cut = p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
        m[f"events in top {int(share * 100)}% (%)"] = 100 * float(w[(p >= cut) & y].sum() / w[y].sum())
    slope, intercept = cal_slope_intercept(y, p, w)
    g = np.empty(len(p), int); g[o] = np.minimum((10 * cw / cw[-1]).astype(int), 9)   # weighted deciles
    ece = sum(abs(np.sum(w[g == k] * y[g == k]) - np.sum(w[g == k] * p[g == k])) for k in range(10)) / w.sum()
    m.update({"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p, sample_weight=w),
              "prevalence": float(np.sum(w * y) / w.sum()), "Brier": float(np.sum(w * (p - y) ** 2) / w.sum()),
              "calibration slope": slope, "calibration intercept": intercept,
              "observed/expected": float(np.sum(w * y) / np.sum(w * p)), "ECE": float(ece)})
    return m

def boot_ci(y, p, w, keys, seed):
    rng = np.random.default_rng(seed); pos, neg = np.where(y)[0], np.where(~y)[0]; res = []
    for _ in range(B):
        ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        res.append({k: v for k, v in metrics(y[ix], p[ix], w[ix]).items() if k in keys})
    r = pd.DataFrame(res)
    return r.quantile(0.025), r.quantile(0.975)

def fit_calibrated(name, X, y, w, seed, cols=None):
    """Fit on 80% of the training part and Platt-recalibrate on the other 20% (inside training)."""
    cols = cols or PREDICTORS
    X = X[cols]
    X_f, X_c, y_f, y_c, w_f, w_c = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=seed)
    m = make_model(name); step = m.steps[-1][0]
    m.fit(X_f, y_f, **{f"{step}__sample_weight": w_f})
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(m.predict_proba(X_c)[:, 1]).reshape(-1, 1), y_c, sample_weight=w_c)
    return lambda Xn: platt.predict_proba(logit(m.predict_proba(Xn[cols])[:, 1]).reshape(-1, 1))[:, 1]


f = fit_calibrated(primary, dev[PREDICTORS], dev["label_main"].to_numpy(bool), dev["w"].to_numpy(), SEED)
p_t = f(test[PREDICTORS]); y_t = test["label_main"].to_numpy(bool); w_t = test["w"].to_numpy()
lm_t = test["at_risk_48h"].to_numpy(bool); y48 = test["label_day2_27"].to_numpy(bool)
show("Temporal test 2022-2023 (check: same model as the temporal validation of notebook 05)",
     pd.Series({"AUROC": round(roc_auc_score(y_t, p_t), 4), "births (weighted)": int(round(w_t.sum())),
                "neonatal deaths": int(y_t.sum())}))

## E10 · Workload — how many alerts per death found

Seven operating points, for all live births and for newborns alive at day 2.

| Result | What is done | In this model |
|---|---|---|
| Alerts per death found ≤ 10 with sensitivity ≥ 0.80 | Recommended point for a priority assessment | Risk ≥ 1%: 19 per 1,000, sensitivity 0.82, 8.5 alerts |
| Lower capacity | Top 1% of risk | 10 per 1,000, sensitivity 0.75 |

In [ ]:
# Workload (framework, S10): for a few operating points, how many births are flagged, how many deaths are found
# and how many alerts are needed per death found (1 / PPV). Two populations: all live births at birth + 5 minutes,
# and the newborns alive at the start of day 2 (48-hour landmark), where preventive care has time to act.
def operating_points(y, p, w, label):
    rows = []; n = w.sum(); deaths = w[y].sum()
    o = np.argsort(-p); cw = np.cumsum(w[o])
    points = [("top 1%", None, 0.01), ("top 2%", None, 0.02), ("top 5%", None, 0.05), ("top 10%", None, 0.10),
              ("risk >= 1%", 0.01, None), ("risk >= 2%", 0.02, None), ("risk >= 5%", 0.05, None)]
    for name, thr, share in points:
        cut = thr if thr is not None else p[o][np.searchsorted(cw, share * n)]
        flag = p >= cut
        tp, fp = w[flag & y].sum(), w[flag & ~y].sum(); fn, tn = w[~flag & y].sum(), w[~flag & ~y].sum()
        rows.append({"population": label, "operating point": name, "risk cut": round(float(cut), 4),
                     "% flagged": round(100 * (tp + fp) / n, 2), "flagged per 1,000 births": round(1000 * (tp + fp) / n, 1),
                     "sensitivity": round(tp / deaths, 3), "specificity": round(tn / (tn + fp), 4),
                     "PPV": round(tp / (tp + fp), 4), "alerts per death found": round((tp + fp) / tp, 1) if tp else None,
                     "deaths found per 1,000 births": round(1000 * tp / n, 2),
                     "net benefit": round(tp / n - fp / n * cut / (1 - cut), 5)})
    return rows
wk = operating_points(y_t, p_t, w_t, "all live births (birth + 5 min)") + \
     operating_points(y48[lm_t], p_t[lm_t], w_t[lm_t], "alive at the start of day 2 (deaths on days 2-27)")
show("Workload at several operating points (temporal test 2022-2023, weighted to the full population)",
     pd.DataFrame(wk).set_index(["population", "operating point"]))

## E10 · Equity — performance by subgroup

Mother's race and origin, payment, plurality and cohort; gestational age is only described.

| Result | What is done | In this model |
|---|---|---|
| AUROC more than 0.05 below the total | Flag: the model should not be used in that group without local validation | None |
| O/E outside 0.80–1.25 | Calibration flag: recalibrate or review data quality | Unknown race: 1.45 |

In [ ]:
# Subgroups (framework, S10): does the model work as well across groups? Discrimination, calibration (observed /
# expected, calibration slope) and sensitivity at the overall top-10% cut, on the temporal test. Reporting flags
# fixed before running, for the equity groups (race and Hispanic origin, payment) and plurality and cohort: AUROC more
# than 0.05 below the whole population, or observed/expected outside 0.80-1.25. Gestational-age bands are descriptive
# only: gestational age is one of the strongest predictors, so the AUROC within a band is lower by construction.
o = np.argsort(-p_t); cw = np.cumsum(w_t[o]); cut10 = p_t[o][np.searchsorted(cw, 0.10 * w_t.sum())]
T = test
race = np.select([T["race_black"] == 1, T["race_hispanic"] == 1, T["race_asian"] == 1, T["race_other"] == 1,
                  T[["race_black", "race_hispanic", "race_asian", "race_other"]].isna().any(axis=1)],
                 ["Black", "Hispanic", "Asian", "Other", "unknown"], "White")
pay = np.select([T["pay_medicaid"] == 1, T["pay_self"] == 1, T["pay_other"] == 1,
                 T[["pay_medicaid", "pay_self", "pay_other"]].isna().any(axis=1)],
                ["Medicaid", "self-pay", "other", "unknown"], "private insurance")
ga = T["gestational_age"].to_numpy()
ga_band = np.select([ga < 28, ga < 32, ga < 37], ["22-27 weeks", "28-31 weeks", "32-36 weeks"], "37 weeks or more")
plural = np.where(T["triplet_or_more"] == 1, "triplet or more", np.where(T["twin"] == 1, "twin", "singleton"))
groups = {"mother's race and Hispanic origin": race, "payment of the delivery": pay, "gestational age": ga_band,
          "plurality": plural, "cohort": T["period"].astype(str).to_numpy()}
base_auc = roc_auc_score(y_t, p_t)
rows = []
for gname, g in groups.items():
    for level in sorted(pd.unique(g)):
        m = g == level; yy, pp, ww = y_t[m], p_t[m], w_t[m]
        if yy.sum() < 10 or (~yy).sum() < 10:
            rows.append({"group": gname, "level": level, "births (weighted)": int(round(ww.sum())), "deaths": "<10"}); continue
        auc = roc_auc_score(yy, pp); oe = float(np.sum(ww * yy) / np.sum(ww * pp))
        slope, _ = cal_slope_intercept(yy, pp, ww)
        flag = []
        if gname != "gestational age":
            if auc < base_auc - 0.05: flag.append("AUROC")
            if not 0.80 <= oe <= 1.25: flag.append("O/E")
        rows.append({"group": gname, "level": level, "births (weighted)": int(round(ww.sum())), "deaths": int(yy.sum()),
                     "deaths per 1,000": round(1000 * np.sum(ww * yy) / ww.sum(), 2), "AUROC": round(auc, 4),
                     "observed/expected": round(oe, 3), "calibration slope": round(slope, 3),
                     "sensitivity at top-10% cut": round(float(np.sum(ww * (yy & (pp >= cut10))) / np.sum(ww * yy)), 3),
                     "% flagged at top-10% cut": round(100 * float(np.sum(ww * (pp >= cut10)) / ww.sum()), 1),
                     "flag": ", ".join(flag) or "—"})
show(f"Performance by subgroup (temporal test; whole-population AUROC {base_auc:.4f}; flags: AUROC < "
     f"{base_auc - 0.05:.3f} or O/E outside 0.80-1.25; gestational age descriptive only)", pd.DataFrame(rows).set_index(["group", "level"]))

## E10 · Equity — 95% confidence intervals by subgroup

Stratified bootstrap inside each subgroup (TRIPOD+AI item 23a). A flag is supported when the whole interval is beyond its limit.

| Result | What is done | In this model |
|---|---|---|
| Interval entirely beyond the limit | Flag supported | On running 06 |
| Interval reaches the limit | Flag reported as not confirmed (imprecise estimate) | — |

In [ ]:
# 95% confidence intervals of the subgroup results (TRIPOD+AI item 23a): stratified bootstrap inside each subgroup
# (deaths and other births resampled separately, B resamples, as the other intervals of notebooks 05 and 06), on the
# same predictions of the temporal test. Reading of each flag: supported when the whole interval is beyond the limit
# (AUROC upper limit below the flag limit, or O/E interval entirely outside 0.80-1.25).
rng = np.random.default_rng(SEED + 13); ci_rows = []
for gname, g in groups.items():
    for level in sorted(pd.unique(g)):
        m = g == level; yy, pp, ww = y_t[m], p_t[m], w_t[m]
        if yy.sum() < 10 or (~yy).sum() < 10:
            continue
        pos, neg = np.where(yy)[0], np.where(~yy)[0]; bs = []
        for _ in range(B):
            ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
            bs.append((roc_auc_score(yy[ix], pp[ix]), np.sum(ww[ix] * yy[ix]) / np.sum(ww[ix] * pp[ix])))
        bs = np.array(bs)
        auc, oe = roc_auc_score(yy, pp), float(np.sum(ww * yy) / np.sum(ww * pp))
        a_lo, a_hi = np.percentile(bs[:, 0], [2.5, 97.5]); o_lo, o_hi = np.percentile(bs[:, 1], [2.5, 97.5])
        if gname == "gestational age":
            reading = "descriptive only"
        else:
            notes = []
            if auc < base_auc - 0.05:
                notes.append("AUROC flag supported by the 95% CI" if a_hi < base_auc - 0.05 else "AUROC flag not confirmed: the 95% CI reaches the limit")
            if not 0.80 <= oe <= 1.25:
                notes.append("O/E flag supported by the 95% CI" if (o_hi < 0.80 or o_lo > 1.25) else "O/E flag not confirmed: the 95% CI reaches the limit")
            reading = "; ".join(notes) or "no flag"
        ci_rows.append({"group": gname, "level": level, "deaths": int(yy.sum()), "AUROC": round(auc, 4),
                        "AUROC 2.5%": round(a_lo, 4), "AUROC 97.5%": round(a_hi, 4), "observed/expected": round(oe, 3),
                        "observed/expected 2.5%": round(o_lo, 3), "observed/expected 97.5%": round(o_hi, 3), "reading": reading})
show(f"Performance by subgroup: 95% confidence intervals (stratified bootstrap inside each subgroup, {B} resamples; "
     f"temporal test 2022-2023)", pd.DataFrame(ci_rows).set_index(["group", "level"]))


## E10 · Original objective of the thesis — best sensitivity-specificity pair

Computed although it does not decide in the framework.

| Result | What is done | In this model |
|---|---|---|
| Some threshold gives both ≥ 0.75 | The original objective is met | Yes: 0.904 at the balanced point |

In [ ]:
# Original objective of the thesis (sensitivity and specificity both above 0.75): the best pair any threshold can
# reach. Weighted to the full population; temporal test 2022-2023. Descriptive only.
def best_pairs(y, p, w):
    o = np.argsort(-p); yo, wo, po = y[o], w[o], p[o]
    tp = np.cumsum(wo * yo); fp = np.cumsum(wo * ~yo)
    sens = tp / tp[-1]; spec = 1 - fp / fp[-1]
    last = np.r_[po[1:] != po[:-1], True]          # evaluate only at the end of tied scores
    sens, spec, po = sens[last], spec[last], po[last]
    i = np.argmax(np.minimum(sens, spec))
    j = np.where(sens >= 0.75)[0][0]; k = np.where(spec >= 0.75)[0][-1]
    return {"balanced point: sensitivity": round(float(sens[i]), 3), "balanced point: specificity": round(float(spec[i]), 3),
            "balanced point: risk cut": round(float(po[i]), 4),
            "specificity when sensitivity >= 0.75": round(float(spec[j]), 3),
            "sensitivity when specificity >= 0.75": round(float(sens[k]), 3),
            "some threshold gives both >= 0.75": bool(((sens >= 0.75) & (spec >= 0.75)).any())}
show("Original objective: best sensitivity-specificity pairs (temporal test 2022-2023)", pd.DataFrame({
    "all live births": best_pairs(y_t, p_t, w_t), "alive at the start of day 2": best_pairs(y48[lm_t], p_t[lm_t], w_t[lm_t])}))

## E10 · What drives the risk — SHAP contribution of each predictor (descriptive)

Orients the prevention strategy; it is not causal.

| Result | What is done | In this model |
|---|---|---|
| High weight of the condition at birth | The model serves to act quickly; the prevention that lowers the risk is antenatal | Apgar, weight and gestational age: 68%; care: 13.5% |

In [ ]:
# What drives the risk (to orient the prevention strategy): mean absolute SHAP contribution of each predictor
# in the temporal test (XGBoost), on the log-odds scale. Descriptive, not causal: factors related to prenatal or
# delivery care (where a health entity can act) are marked.
MODIFIABLE = {"no_prenatal_care", "care_after_first_trimester", "prenatal_visits", "smoked_during_pregnancy", "bmi",
              "weight_gain", "antenatal_steroids", "born_out_of_hospital", "attendant_midwife", "attendant_other",
              "induction", "delivery_cesarean", "delivery_forceps", "delivery_vacuum", "wic", "syphilis", "gonorrhea",
              "chlamydia", "hepatitis_b", "hepatitis_c", "chorioamnionitis", "antibiotics_in_labor", "interval_lt18"}
if primary == "XGB":
    import xgboost as xgb
    model = make_model(primary); step = model.steps[-1][0]
    model.fit(dev[PREDICTORS], dev["label_main"].to_numpy(bool), **{f"{step}__sample_weight": dev["w"].to_numpy()})
    Xt = model[:-1].transform(test[PREDICTORS])
    contrib = model[-1].get_booster().predict(xgb.DMatrix(Xt), pred_contribs=True)[:, :-1]
    imp = pd.Series(np.average(np.abs(contrib), axis=0, weights=w_t), index=PREDICTORS).sort_values(ascending=False)
    tab = pd.DataFrame({"mean |SHAP| (log-odds)": imp.round(4),
                        "share of total (%)": (100 * imp / imp.sum()).round(1),
                        "related to prenatal or delivery care": ["yes" if v in MODIFIABLE else "" for v in imp.index]})
    show("Contribution of each predictor (top 25; temporal test, weighted)", tab.head(25))
    show("Share of the total contribution by type of predictor (%)", pd.Series({
        "gestational age and birthweight": round(float(tab.loc[REFERENCE, "share of total (%)"].sum()), 1),
        "related to prenatal or delivery care": round(float(tab.loc[tab.index.isin(MODIFIABLE), "share of total (%)"].sum()), 1),
        "other": round(float(100 - tab.loc[REFERENCE, "share of total (%)"].sum() - tab.loc[tab.index.isin(MODIFIABLE), "share of total (%)"].sum()), 1)}))

## Figures · ROC curves and classification report of the final model

ROC curves on the temporal test, one subplot per population (all live births and alive at the start of day 2), and, at the recommended cut (risk ≥ 1%) and at the top 10% of risk, the classification report (precision, recall, F1, accuracy, macro average) next to the confusion matrix, weighted to the population.

In [ ]:
# Figure helpers (shared by the figure and summary cells). Curves use the out-of-fold or test predictions;
# confusion matrices are weighted to the full population (non-events carry weight 1/FRACTION).
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

def plot_roc(curves, title, fname):
    """curves: list of (label, y, p). One subplot per curve (algorithm or population), so the curves do not overlap.
    AUROC does not depend on the case-control weights."""
    n = len(curves); ncol = min(3, n); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 4.4 * nrow), squeeze=False)
    for k, (label, yy, pp) in enumerate(curves):
        ax = axes[k // ncol][k % ncol]
        fpr, tpr, _ = roc_curve(yy, pp)
        ax.plot(fpr, tpr, lw=1.8, color="#2F6DB5")
        ax.fill_between(fpr, tpr, alpha=0.08, color="#2F6DB5")
        ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=1)
        ax.set_title(f"{translate(label, LANG)}\nAUROC {roc_auc_score(yy, pp):.3f}", fontsize=10)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.set_aspect("equal")
        ax.set_xlabel(translate("1 - specificity (false positive rate)", LANG), fontsize=9)
        ax.set_ylabel(translate("Sensitivity (true positive rate)", LANG), fontsize=9)
    for k in range(n, nrow * ncol):
        axes[k // ncol][k % ncol].axis("off")
    fig.suptitle(translate(title, LANG), fontsize=12)
    fig.tight_layout(); fig.savefig(FIG_DIR / (fname.replace(".png", "_es.png") if LANG == "es" else fname), dpi=200); plt.show()

def cut_top(p, w, share):
    """Risk cut that flags the given share of the weighted population."""
    o = np.argsort(-p); cw = np.cumsum(w[o])
    return float(p[o][np.searchsorted(cw, share * cw[-1])])

def threshold_metrics(y, p, w, cut):
    """Accuracy, F1 and the confusion-matrix rates at a declared risk cut (weighted to the population)."""
    flag = p >= cut
    tp, fp = w[flag & y].sum(), w[flag & ~y].sum(); fn, tn = w[~flag & y].sum(), w[~flag & ~y].sum()
    sens, ppv = tp / (tp + fn), tp / (tp + fp)
    return {"risk cut": round(cut, 4), "TP": int(round(tp)), "FP": int(round(fp)), "FN": int(round(fn)), "TN": int(round(tn)),
            "accuracy": round(float((tp + tn) / (tp + fp + fn + tn)), 4), "F1": round(float(2 * sens * ppv / (sens + ppv)), 4),
            "sensitivity": round(float(sens), 4), "specificity": round(float(tn / (tn + fp)), 4),
            "PPV": round(float(ppv), 4), "NPV": round(float(tn / (tn + fn)), 5)}

def plot_cm(panels, fname):
    """panels: list of (title, y, p, w, cut). For each panel, on the left the classification report of the two classes
    (precision, recall, F1, accuracy and macro average) as a heatmap and, on the right, the confusion matrix. Counts are
    weighted to the full population (non-events carry weight 1/FRACTION); class 1 = neonatal death."""
    from sklearn.metrics import ConfusionMatrixDisplay
    fig, axes = plt.subplots(len(panels), 2, figsize=(12.5, 4.6 * len(panels)), squeeze=False,
                             gridspec_kw={"width_ratios": [1.35, 1]})
    for row, (title, yy, pp, ww, cut) in enumerate(panels):
        m = threshold_metrics(yy, pp, ww, cut)
        tp, fp, fn, tn = m["TP"], m["FP"], m["FN"], m["TN"]
        f1 = lambda a, b: 2 * a * b / (a + b) if (a + b) else 0.0
        p0, r0 = tn / (tn + fn), tn / (tn + fp)
        p1, r1 = tp / (tp + fp), tp / (tp + fn)
        acc = (tp + tn) / (tp + fp + fn + tn)
        rep = np.array([[p0, r0, f1(p0, r0)], [p1, r1, f1(p1, r1)], [acc, acc, acc],
                        [(p0 + p1) / 2, (r0 + r1) / 2, (f1(p0, r0) + f1(p1, r1)) / 2]])
        ax = axes[row][0]
        ax.imshow(rep, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
        for i in range(rep.shape[0]):
            for j in range(rep.shape[1]):
                ax.text(j, i, f"{rep[i, j]:.2f}", ha="center", va="center", fontsize=10,
                        color="white" if rep[i, j] > 0.6 else "black")
        ax.set_xticks(range(3), [translate(c, LANG) for c in ("precision", "recall", "f1-score")])
        ax.set_yticks(range(4), ["0", "1", translate("accuracy", LANG), translate("macro avg", LANG)])
        ax.set_xlabel(translate("Metrics", LANG)); ax.set_ylabel(translate("Classes", LANG))
        ax.set_title(translate(title, LANG) + f" ({translate('risk cut', LANG)} {cut:.4f})", fontsize=9)
        ax2 = axes[row][1]
        ConfusionMatrixDisplay(confusion_matrix=np.array([[tn, fp], [fn, tp]]), display_labels=[0, 1]).plot(
            ax=ax2, cmap="Blues", colorbar=False, values_format=",d")
        ax2.set_xlabel(translate("Predicted label", LANG)); ax2.set_ylabel(translate("True label", LANG))
        ax2.set_title(translate("Confusion matrix", LANG), fontsize=9)
    fig.tight_layout(); fig.savefig(FIG_DIR / (fname.replace(".png", "_es.png") if LANG == "es" else fname), dpi=200); plt.show()

plot_roc([("all live births", y_t, p_t), ("alive at the start of day 2 (deaths on days 2-27)", y48[lm_t], p_t[lm_t])],
         "Temporal test 2022-2023 (model fitted once on 2016-2021)", "06_roc_temporal.png")
plot_cm([("Recommended cut: risk >= 1%", y_t, p_t, w_t, 0.01), ("Top 10% of risk", y_t, p_t, w_t, cut10)], "06_confusion_matrices.png")

## Notebook summary

Operating points with accuracy, F1, sensitivity, specificity and PPV; equity and what drives the risk.

In [ ]:
# Summary of notebook 06 (stage S10)
op = pd.DataFrame({"risk >= 1% (recommended)": threshold_metrics(y_t, p_t, w_t, 0.01),
                   "top 1% of risk": threshold_metrics(y_t, p_t, w_t, cut_top(p_t, w_t, 0.01)),
                   "top 10% of risk": threshold_metrics(y_t, p_t, w_t, cut10)}).T
op["flagged per 1,000 births"] = (1000 * (op["TP"] + op["FP"]) / w_t.sum()).round(1)
op["alerts per death found"] = ((op["TP"] + op["FP"]) / op["TP"]).round(1)
show("Summary: operating points (temporal test 2022-2023)", op)
sub = pd.DataFrame(rows).dropna(subset=["AUROC"]) if "AUROC" in pd.DataFrame(rows) else pd.DataFrame(rows)
eq = sub[sub["group"] != "gestational age"]
show("Summary: model, equity and drivers", pd.Series({
    "AUROC, temporal test": round(base_auc, 4),
    "AUROC range across equity groups": f"{eq['AUROC'].min():.3f}-{eq['AUROC'].max():.3f}",
    "subgroups flagged": ", ".join(f"{g}: {l} ({f})" for g, l, f in eq[eq["flag"] != "—"][["group", "level", "flag"]].itertuples(index=False)) or "none",
    "share of risk: gestational age and birthweight (%)": float(tab.loc[REFERENCE, "share of total (%)"].sum()) if primary == "XGB" else None,
    "share of risk: related to prenatal or delivery care (%)": float(tab.loc[tab.index.isin(MODIFIABLE), "share of total (%)"].sum()) if primary == "XGB" else None}))
write_outputs(OUT_FILE, "Stage S10 (from classification to action) of the neonatal model, temporal test 2022-2023")
translate(op, LANG)

## E11 · Reporting — what is delivered and where it is

A stage without cells: the report is built from what notebooks 01 to 06 produce, following TRIPOD+AI.

| Result | What is done | In this model |
|---|---|---|
| Aggregate outputs | `data/outputs_01` to `outputs_06` (+ `outputs_05b`) | Done |
| Outcome card and decision log | `docs/outcome_card_neonatal_nchs.md`, `docs/decision_G4_C1_2026-11-23.md` | Done |
| Guide for use (S10) | `docs/e10_action_neonatal.md` | Done |
| Figures | `figures/04_*.png` to `06_*.png` | On running 04 to 06 |
| Final model | `models/` (notebook 05): joblib pipeline, XGBoost JSON and model card; use in `libraries/neonatal_model.py` | On running 05 |
| Software versions | `requirements.txt` with the exact versions of the environment that ran the notebooks | Done |
| TRIPOD+AI checklist and thesis chapter | Written from the above | Pending |